# Arabic Phishing Detection Project

هذا الدفتر **Google Colab**، ويشمل:

- قراءة ملفات **Train / Validation / Test**
- تنظيف وفحص البيانات
- تدريب وتقييم **SVM**
- تدريب وتقييم **AraBERT**


- مقارنة النتائج النهائية مع نماذج **Hybird**



- حفظ النتائج والملفات النهائية

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece arabert

# تحميل المكتبات وتهيأءة البيئة للمشروع

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

print("Seed fixed:", SEED)
print("CUDA available:", torch.cuda.is_available())

## رفع الملفات إلى Colab

In [ ]:
TRAIN_FILE = "merged_train_enhanced.csv"
VALID_FILE = "merged_valid_enhanced.csv"
TEST_FILE  = "merged_test_enhanced.csv"

train_df = pd.read_csv(TRAIN_FILE)
valid_df = pd.read_csv(VALID_FILE)
test_df  = pd.read_csv(TEST_FILE)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape :", test_df.shape)


In [ ]:
display(train_df.head())
display(valid_df.head())
display(test_df.head())

print("Train label distribution:")
print(train_df["label"].value_counts(dropna=False), "\n")

print("Valid label distribution:")
print(valid_df["label"].value_counts(dropna=False), "\n")

print("Test label distribution:")
print(test_df["label"].value_counts(dropna=False))

## تنظيف مبدئي للبيانات

In [ ]:
REQUIRED_COLUMNS = ["text", "label"]

def basic_clean_df(df, name="df"):
    df = df.copy()

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            raise ValueError(f"{name} is missing required column: {col}")

    df["text"] = df["text"].astype(str).fillna("").str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)

    df = df[df["text"].str.len() > 0].copy()
    df = df[df["label"].isin([0, 1])].copy()
    df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    return df

train_df = basic_clean_df(train_df, "train_df")
valid_df = basic_clean_df(valid_df, "valid_df")
test_df  = basic_clean_df(test_df, "test_df")

print("After basic cleaning:")
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

## استكشاف وتحليل البيانات (EDA)
> تحليل بصري لتوزيع الفئات وأطوال النصوص

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Validation', valid_df), ('Test', test_df)]):
    counts = df['label'].value_counts().sort_index()
    ax.bar(['Legitimate (0)', 'Phishing (1)'], counts.values, color=['#2196F3', '#F44336'])
    ax.set_title(f'{name} Label Distribution')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Text length distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Validation', valid_df), ('Test', test_df)]):
    df['text_len'] = df['text'].astype(str).apply(len)
    ax.hist(df[df['label']==0]['text_len'], bins=30, alpha=0.6, label='Legitimate', color='#2196F3')
    ax.hist(df[df['label']==1]['text_len'], bins=30, alpha=0.6, label='Phishing', color='#F44336')
    ax.set_title(f'{name} Text Length')
    ax.set_xlabel('Characters')
    ax.legend()
plt.tight_layout()
plt.show()

print('Average text lengths:')
for name, df in [('Train', train_df), ('Valid', valid_df), ('Test', test_df)]:
    print(f'  {name}: {df["text"].astype(str).apply(len).mean():.0f} chars')


### تحليل نتائج الاستكشاف (EDA Analysis)

**توزيع الفئات:**
- بيانات التدريب متوازنة تمامًا (1575 لكل فئة)، مما يزيل التحيز أثناء التعلم.
- بيانات التحقق متوازنة تقريبًا (338 مقابل 337).
-  بيانات الاختبار غير متوازنة (473 سليمة مقابل 202 تصيّد بنسبة 70:30).


**أطوال النصوص:**
- متوسط أطوال نصوص التدريب (175 حرفًا) أقل من التحقق (232) والاختبار (204).
- يشير ذلك إلى أن بيانات التحقق والاختبار تحتوي على نصوص أكثر تعقيدًا.

## Normalization للنصوص لاكتشاف التسرب الحقيقي
> نستخدمها فقط لفحص التطابق، وليس كبديل مباشر للنص الأصلي في التدريب.

In [ ]:
def normalize_text(text):
    text = str(text).strip().lower()

    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' <EMAIL> ', text)
    text = re.sub(r'@\w+', ' <USER> ', text)
    text = re.sub(r'#\w+', ' <HASHTAG> ', text)

    text = re.sub(r'[٠-٩]', '0', text)
    text = re.sub(r'\d+', '0', text)

    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)

    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    text = re.sub(r'[^\w\s<>]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

for df in [train_df, valid_df, test_df]:
    df["norm_text"] = df["text"].apply(normalize_text)

display(train_df[["text", "norm_text"]].head())

## إزالة التسرب الحقيقي فقط
نزيل فقط **exact duplicates after normalization** بين `Train` و `Validation/Test`.

In [ ]:
train_norm_set = set(train_df["norm_text"])

valid_clean = valid_df[~valid_df["norm_text"].isin(train_norm_set)].copy()
test_clean  = test_df[~test_df["norm_text"].isin(train_norm_set)].copy()

valid_norm_set = set(valid_clean["norm_text"])
test_clean = test_clean[~test_clean["norm_text"].isin(valid_norm_set)].copy()

valid_clean = valid_clean.reset_index(drop=True)
test_clean  = test_clean.reset_index(drop=True)

print("After exact leakage removal:")
print("Train:", len(train_df))
print("Valid clean:", len(valid_clean))
print("Test clean :", len(test_clean))

In [ ]:
if len(valid_clean) == 0:
    raise ValueError("Validation set became empty after exact leakage removal.")

if len(test_clean) == 0:
    raise ValueError("Test set became empty after exact leakage removal.")

print("Validation and Test are safe to use.")

## حفظ النسخ النظيفة

In [ ]:
train_clean = train_df.copy()

train_clean.to_csv("train_clean_final.csv", index=False, encoding="utf-8-sig")
valid_clean.to_csv("valid_clean_final.csv", index=False, encoding="utf-8-sig")
test_clean.to_csv("test_clean_final.csv", index=False, encoding="utf-8-sig")

print("Saved cleaned files.")

# الجزء الأول: SVM

In [ ]:
sample_train = train_clean["text"].astype(str).tolist()
y_train = train_clean["label"].astype(int).tolist()

sample_valid = valid_clean["text"].astype(str).tolist()
y_valid = valid_clean["label"].astype(int).tolist()

sample_test = test_clean["text"].astype(str).tolist()
y_test = test_clean["label"].astype(int).tolist()

print("SVM sizes:")
print("Train:", len(sample_train))
print("Valid:", len(sample_valid))
print("Test :", len(sample_test))

In [ ]:
classes = np.array(sorted(train_clean["label"].unique()))
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_clean["label"].values
)

class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
print("Class weights:", class_weight_dict)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

# Define the base pipeline
svm_base_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        min_df=2,
        max_df=0.90,
        sublinear_tf=True,
        strip_accents=None,
        lowercase=True
    )),
    ("clf", LinearSVC(
        class_weight="balanced",
        random_state=42,
        max_iter=5000
    ))
])

# Hyperparameter tuning with GridSearchCV
param_grid = {
    "clf__C": [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0],
    "tfidf__ngram_range": [ (1,3)]
}

grid_search = GridSearchCV(
    svm_base_pipeline, param_grid,
    cv=5, scoring="f1", n_jobs=-1, verbose=1
)

grid_search.fit(sample_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 4))

svm_pipeline = grid_search.best_estimator_
print("\nOptimized SVM Pipeline Ready")

In [ ]:
# No need to re-fit - GridSearchCV already trained the best model
valid_pred_svm = svm_pipeline.predict(sample_valid)

print("=== SVM Validation Results ===")
print("Accuracy :", accuracy_score(y_valid, valid_pred_svm))
print("Precision:", precision_score(y_valid, valid_pred_svm))
print("Recall   :", recall_score(y_valid, valid_pred_svm))
print("F1-score :", f1_score(y_valid, valid_pred_svm))

print("\nClassification Report (Validation):")
print(classification_report(y_valid, valid_pred_svm, digits=4))

In [ ]:
test_pred_svm = svm_pipeline.predict(sample_test)

print("=== SVM Test Results ===")
print("Accuracy :", accuracy_score(y_test, test_pred_svm))
print("Precision:", precision_score(y_test, test_pred_svm))
print("Recall   :", recall_score(y_test, test_pred_svm))
print("F1-score :", f1_score(y_test, test_pred_svm))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred_svm, digits=4))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scores for Soft Voting
svm_scores = svm_pipeline.decision_function(sample_test)

scaler = MinMaxScaler()
svm_probs = scaler.fit_transform(
    np.array(svm_scores).reshape(-1,1)
).ravel()

# Save file
svm_results = pd.DataFrame({
    "text": sample_test,
    "true_label": y_test,
    "svm_pred": test_pred_svm,
    "svm_prob": svm_probs
})

svm_results.to_csv("svm_test_predictions.csv", index=False, encoding="utf-8-sig")

In [ ]:
cm = confusion_matrix(y_test, test_pred_svm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Phishing"])
disp.plot(cmap="Blues")
plt.title("SVM Confusion Matrix - Test")
plt.show()

### SVM ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

# SVM decision scores for ROC
svm_scores = svm_pipeline.decision_function(sample_test)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_scores)
auc_svm = auc(fpr_svm, tpr_svm)

plt.figure(figsize=(7, 5))
plt.plot(fpr_svm, tpr_svm, color='#1565C0', lw=2, label=f'SVM (AUC = {auc_svm:.4f})')
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('SVM ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f'SVM AUC: {auc_svm:.4f}')


# الجزء الثاني: AraBERT

In [ ]:
from arabert.preprocess import ArabertPreprocessor

MODEL_NAME = "aubmindlab/bert-base-arabertv02"

arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)

print("Model:", MODEL_NAME)

In [ ]:
train_bert = train_clean[["text", "label"]].rename(columns={"label": "labels"}).copy()
valid_bert = valid_clean[["text", "label"]].rename(columns={"label": "labels"}).copy()
test_bert  = test_clean[["text", "label"]].rename(columns={"label": "labels"}).copy()

train_bert["text"] = train_bert["text"].astype(str).apply(arabert_prep.preprocess)
valid_bert["text"] = valid_bert["text"].astype(str).apply(arabert_prep.preprocess)
test_bert["text"]  = test_bert["text"].astype(str).apply(arabert_prep.preprocess)

print(train_bert.shape, valid_bert.shape, test_bert.shape)

In [ ]:
train_dataset = Dataset.from_pandas(train_bert, preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_bert, preserve_index=False)
test_dataset  = Dataset.from_pandas(test_bert, preserve_index=False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
MAX_LEN = 128  # Reduced from 256 to fit in local GPU memory

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

In [ ]:
columns_to_return = ["input_ids", "attention_mask", "labels"]

if "token_type_ids" in train_dataset.column_names:
    columns_to_return.append("token_type_ids")

train_dataset.set_format(type="torch", columns=columns_to_return)
valid_dataset.set_format(type="torch", columns=columns_to_return)
test_dataset.set_format(type="torch", columns=columns_to_return)

print("Dataset format ready.")

In [ ]:
# Free GPU memory before loading model
import gc
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./arabert_results_v2",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    per_device_train_batch_size=16
    ,
    per_device_eval_batch_size=32,

    gradient_accumulation_steps=8,
    lr_scheduler_type = "cosine",
    label_smoothing_factor = 0.1,

    num_train_epochs=5,

    learning_rate = 2e-5,
    weight_decay=0.01,
    warmup_steps=197, # Calculated as 0.1 * (len(train_dataset) / per_device_train_batch_size) * num_train_epochs

    save_total_limit=2,
    report_to="none",

    seed=42,

    gradient_checkpointing=True,  # Saves VRAM by recomputing activations

    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
torch.cuda.empty_cache()
trainer.train()

In [ ]:
# ==================================
# Save Final AraBERT Model
# ==================================

SAVE_PATH = "/content/arabert_phishing_model_finalWeb"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("AraBERT model saved successfully at:", SAVE_PATH)

In [ ]:
# ==================================
# Zip + Download Model
# ==================================

!zip -r arabert_phishing_model_final.zip /content/arabert_phishing_model_finalWeb

from google.colab import files
files.download("arabert_phishing_model_final.zip")

### حفظ النموذج المُدرَّب

In [ ]:
# Save the fine-tuned AraBERT model
trainer.save_model('./arabert_phishing_model_final')
tokenizer.save_pretrained('./arabert_phishing_model_final')
print('AraBERT model saved to ./arabert_phishing_model_final')


In [ ]:
from sklearn.metrics import f1_score
import numpy as np
import torch

valid_pred_output = trainer.predict(valid_dataset)

valid_logits = valid_pred_output.predictions
valid_probs = torch.softmax(torch.tensor(valid_logits), dim=1).numpy()[:,1]
valid_true = valid_pred_output.label_ids

best_threshold = 0.5
best_f1 = 0

for th in np.arange(0.30, 0.71, 0.02):
    preds = (valid_probs >= th).astype(int)
    score = f1_score(valid_true, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = th

print("Best Threshold =", best_threshold)
print("Best Validation F1 =", best_f1)

In [ ]:
valid_results = trainer.evaluate(valid_dataset)
print("AraBERT Validation Results:")
print(valid_results)

In [ ]:
test_results = trainer.evaluate(test_dataset)
print("AraBERT Test Results:")
print(test_results)

In [ ]:
test_pred_output = trainer.predict(test_dataset)

test_logits = test_pred_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()[:,1]

true_labels = test_pred_output.label_ids

pred_labels = (test_probs >= best_threshold).astype(int)

print("Accuracy :", accuracy_score(true_labels, pred_labels))
print("Precision:", precision_score(true_labels, pred_labels))
print("Recall   :", recall_score(true_labels, pred_labels))
print("F1-score :", f1_score(true_labels, pred_labels))

print(classification_report(true_labels, pred_labels, digits=4))

In [ ]:
# ==================================
# AraBERT Predictions + Probabilities
# ==================================

import numpy as np
from scipy.special import softmax

logits = test_pred_output.predictions

# labels
pred_labels = np.argmax(logits, axis=1)

# phishing probability (class 1)
test_probs = softmax(logits, axis=1)[:,1]

print("Predictions ready.")
print("Shape:", pred_labels.shape)

In [ ]:
# ==================================
# Save AraBERT Test Results
# ==================================

bert_results = pd.DataFrame({
    "text": sample_test,
    "true_label": true_labels,
    "arabert_pred": pred_labels,
    "arabert_prob": test_probs
})

bert_results.to_csv(
    "arabert_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("arabert_test_predictions.csv saved.")

In [ ]:
cm_bert = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_bert, display_labels=["Legitimate", "Phishing"])
disp.plot(cmap="Greens")
plt.title("AraBERT Confusion Matrix - Test")
plt.show()

### ROC Curve Comparison

In [ ]:
# AraBERT ROC
fpr_bert, tpr_bert, _ = roc_curve(true_labels, test_probs)
auc_bert = auc(fpr_bert, tpr_bert)

# Combined ROC
plt.figure(figsize=(7, 5))
plt.plot(fpr_svm, tpr_svm, color='#1565C0', lw=2, label=f'SVM (AUC = {auc_svm:.4f})')
plt.plot(fpr_bert, tpr_bert, color='#2E7D32', lw=2, label=f'AraBERT (AUC = {auc_bert:.4f})')
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: SVM vs AraBERT')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# مقارنة نهائية بين النموذجين

In [ ]:
svm_acc = accuracy_score(y_test, test_pred_svm)
svm_prec = precision_score(y_test, test_pred_svm)
svm_rec = recall_score(y_test, test_pred_svm)
svm_f1 = f1_score(y_test, test_pred_svm)

bert_acc = accuracy_score(true_labels, pred_labels)
bert_prec = precision_score(true_labels, pred_labels)
bert_rec = recall_score(true_labels, pred_labels)
bert_f1 = f1_score(true_labels, pred_labels)

comparison_df = pd.DataFrame({
    "Model": ["SVM", "AraBERT"],
    "Accuracy": [svm_acc, bert_acc],
    "Precision": [svm_prec, bert_prec],
    "Recall": [svm_rec, bert_rec],
    "F1-score": [svm_f1, bert_f1]
})

display(comparison_df)

In [ ]:
# Visual comparison bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
svm_vals = [svm_acc, svm_prec, svm_rec, svm_f1]
bert_vals = [bert_acc, bert_prec, bert_rec, bert_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, svm_vals, width, label='SVM', color='#1565C0')
bars2 = ax.bar(x + width/2, bert_vals, width, label='AraBERT', color='#2E7D32')

ax.set_ylabel('Score')
ax.set_title('Model Comparison: SVM vs AraBERT')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.15)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# McNemar's Test for Statistical Significance
from scipy.stats import chi2

svm_correct = (np.array(y_test) == np.array(test_pred_svm))
bert_correct = (np.array(true_labels) == np.array(pred_labels))

a = np.sum(svm_correct & bert_correct)
b = np.sum(svm_correct & ~bert_correct)
c = np.sum(~svm_correct & bert_correct)
d = np.sum(~svm_correct & ~bert_correct)

print('McNemar Contingency Table:')
print(f'  Both Correct:               {a}')
print(f'  SVM Correct, AraBERT Wrong:  {b}')
print(f'  SVM Wrong, AraBERT Correct:  {c}')
print(f'  Both Wrong:                  {d}')

if b + c > 0:
    mcnemar_stat = (abs(b - c) - 1)**2 / (b + c)
    p_value = 1 - chi2.cdf(mcnemar_stat, df=1)
    print(f'\nMcNemar Statistic: {mcnemar_stat:.4f}')
    print(f'P-value: {p_value:.4f}')
    if p_value < 0.05:
        print('Result: Statistically significant (p < 0.05)')
    else:
        print('Result: NOT statistically significant (p >= 0.05)')
else:
    print('Cannot compute: b + c = 0')


In [ ]:
comparison_df.to_csv("model_comparison_results.csv", index=False, encoding="utf-8-sig")

svm_predictions_df = pd.DataFrame({
    "text": sample_test,
    "true_label": y_test,
    "svm_pred": test_pred_svm,
    "svm_prob": svm_probs # Include svm_prob here as well
})
svm_predictions_df.to_csv("svm_test_predictions.csv", index=False, encoding="utf-8-sig")

bert_predictions_df = pd.DataFrame({
    "text": test_clean["text"].tolist(),
    "true_label": true_labels,
    "arabert_pred": pred_labels,
    "arabert_prob": test_probs # Add arabert_prob here
})
bert_predictions_df.to_csv("arabert_test_predictions.csv", index=False, encoding="utf-8-sig")

print("All results saved.")

In [ ]:
# =====================================================
# Hybrid Step 1: Get SVM probabilities on validation set
# =====================================================
from sklearn.preprocessing import MinMaxScaler

svm_valid_scores = svm_pipeline.decision_function(sample_valid)
scaler_valid = MinMaxScaler()
svm_valid_probs = scaler_valid.fit_transform(
    np.array(svm_valid_scores).reshape(-1, 1)
).ravel()

# Also re-scale test scores using same scaler for consistency
svm_test_scaled = scaler_valid.transform(
    np.array(svm_pipeline.decision_function(sample_test)).reshape(-1, 1)
).ravel()

print(" SVM probabilities ready")
print("  Valid sample:", svm_valid_probs[:5].round(4))
print("  Test  sample:", svm_test_scaled[:5].round(4))

In [ ]:
# =====================================================
# Hybrid Step 2: Soft Voting Ensemble (Weighted Average)
# =====================================================
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score)

# valid_probs  = AraBERT probs on validation (from cell 48)
# test_probs   = AraBERT probs on test       (from cell 51/52)
# svm_valid_probs = SVM probs on validation  (cell above)
# svm_test_scaled = SVM probs on test        (cell above)

ensemble_results = []

for svm_w in np.arange(0.1, 1.0, 0.1):
    bert_w = round(1.0 - svm_w, 1)

    # Find best threshold on VALIDATION
    best_th, best_f1_v = 0.5, 0.0
    for th in np.arange(0.30, 0.71, 0.02):
        hybrid_valid = svm_w * svm_valid_probs + bert_w * valid_probs
        preds = (hybrid_valid >= th).astype(int)
        sc = f1_score(valid_true, preds)
        if sc > best_f1_v:
            best_f1_v, best_th = sc, th

    # Evaluate on TEST
    hybrid_test = svm_w * svm_test_scaled + bert_w * test_probs
    hybrid_preds = (hybrid_test >= best_th).astype(int)

    ensemble_results.append({
        "SVM Weight":  round(svm_w, 1),
        "BERT Weight": bert_w,
        "Threshold":   round(best_th, 2),
        "Accuracy":    round(accuracy_score(y_test, hybrid_preds), 4),
        "Precision":   round(precision_score(y_test, hybrid_preds), 4),
        "Recall":      round(recall_score(y_test, hybrid_preds), 4),
        "F1-score":    round(f1_score(y_test, hybrid_preds), 4),
    })

ensemble_df = pd.DataFrame(ensemble_results)
print("=== Soft Voting Ensemble Results ===")
display(ensemble_df.sort_values("F1-score", ascending=False))

best_row = ensemble_df.sort_values("F1-score", ascending=False).iloc[0]
print(f"\nBest: SVM={best_row['SVM Weight']}  BERT={best_row['BERT Weight']}"
      f"  Threshold={best_row['Threshold']}  F1={best_row['F1-score']}")

In [ ]:
# =====================================================
# Hybrid Step 3: Stacking + Full 4-Model Comparison
# =====================================================
from sklearn.linear_model import LogisticRegression

# --- Train Stacking meta-learner on VALIDATION probabilities ---
meta_X_valid = np.column_stack([svm_valid_probs, valid_probs])
meta_X_test  = np.column_stack([svm_test_scaled, test_probs])

meta_clf = LogisticRegression(class_weight="balanced", random_state=42, C=1.0)
meta_clf.fit(meta_X_valid, np.array(valid_true))

stacking_preds = meta_clf.predict(meta_X_test)

stacking_acc  = accuracy_score(y_test, stacking_preds)
stacking_prec = precision_score(y_test, stacking_preds)
stacking_rec  = recall_score(y_test, stacking_preds)
stacking_f1   = f1_score(y_test, stacking_preds)

print("=== Stacking Results ===")
print("Accuracy :", round(stacking_acc,  4))
print("Precision:", round(stacking_prec, 4))
print("Recall   :", round(stacking_rec,  4))
print("F1-score :", round(stacking_f1,   4))
print()
print(classification_report(y_test, stacking_preds, digits=4))

# --- Best Ensemble preds ---
best_row = ensemble_df.sort_values("F1-score", ascending=False).iloc[0]
best_hybrid = (best_row["SVM Weight"] * svm_test_scaled +
               best_row["BERT Weight"] * test_probs)
best_hybrid_preds = (best_hybrid >= best_row["Threshold"]).astype(int)

# --- Full 4-Model Comparison Table ---
full_df = pd.DataFrame({
    "Model":     ["SVM", "AraBERT",
                  f"Hybrid Ensemble (SVM={best_row['SVM Weight']}/BERT={best_row['BERT Weight']})",
                  "Hybrid Stacking (LR)"],
    "Accuracy":  [svm_acc,  bert_acc,
                  accuracy_score(y_test, best_hybrid_preds),  stacking_acc],
    "Precision": [svm_prec, bert_prec,
                  precision_score(y_test, best_hybrid_preds), stacking_prec],
    "Recall":    [svm_rec,  bert_rec,
                  recall_score(y_test, best_hybrid_preds),    stacking_rec],
    "F1-score":  [svm_f1,   bert_f1,
                  f1_score(y_test, best_hybrid_preds),        stacking_f1],
})

print("=== Final 4-Model Comparison ===")
display(full_df)

# --- Bar Chart ---
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
x = np.arange(len(metrics))
width = 0.2
colors = ["#1565C0", "#2E7D32", "#E65100", "#6A1B9A"]

fig, ax = plt.subplots(figsize=(12, 5))
for i, (_, row) in enumerate(full_df.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + (i - 1.5) * width, vals, width,
                  label=row["Model"], color=colors[i])
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.3f}",
                    xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", fontsize=7)

ax.set_ylabel("Score")
ax.set_title("Full Model Comparison: SVM vs AraBERT vs Hybrid")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.show()

# --- Save ---
full_df.to_csv("hybrid_full_comparison.csv", index=False, encoding="utf-8-sig")
print(" Saved hybrid_full_comparison.csv")

## تنزيل النتائج من Colab

In [ ]:
from google.colab import files

files.download("model_comparison_results.csv")
files.download("svm_test_predictions.csv")
files.download("arabert_test_predictions.csv")
files.download("train_clean_final.csv")
files.download("valid_clean_final.csv")
files.download("test_clean_final.csv")